[## Text Processing - Tokenization, Stemming, Lemmatization](https://github.com/rohitg00/ai-engineering-from-scratch/blob/main/phases/05-nlp-foundations-to-advanced/01-text-processing/docs/en.md)

Computers can only do math with numbers, and words aren’t numbers. Before a model can learn anything from text, you have to turn the text into numbers. The passage says this conversion starts with three questions, and each one is a way of getting it wrong.

### Question 1: Where does a word start and end? (tokenization)

Imagine cutting a sentence into LEGO bricks. The model gets bricks, not sentences.

“The cats were running.” might become: `The` | `cats` | `were` | `running` | `.`

It looks easy: cut at spaces. But look at **don’t**. You could cut it two ways:

- One brick: `don't`
- Two bricks: `do` + `n't`

Both are reasonable. The problem is mixing them. Say half your training data is cut the first way and half the second way. The model now sees the same idea as two different things, and it learns a confused picture. The passage calls this splitting the training distribution. Whichever rule you pick, apply it everywhere, every time.

Punctuation has the same problem. Is `.` its own brick, or part of `running.`? If it’s glued on, `running.` and `running` look like two unrelated words.

### Question 2: What is the root of the word? (stemming)

Think of a tree. `running`, `runs` and `runner` are different branches growing from the same root, `run`.

Stemming is a crude chop: cut off the endings and keep what’s left.

- `running` → `run`
- `cats` → `cat`

It’s fast and dumb. It doesn’t know any English, it just follows chopping rules. That’s where it goes wrong. The passage’s example is `organization` and `organ`. One is about a company or group, the other is a body part. A chopper that cuts both down to `organ` has glued two unrelated ideas together. If you then ask a topic model “what are these documents about?”, it sees a pile of stuff about organs. That’s what “topic modeling dies” means.

### Question 3: Which words should count as the same? (lemmatization)

`run`, `running` and `ran` are the same action in different tenses. Stemming can’t connect `ran` to `run`, because chopping the end off `ran` gets you nowhere. Lemmatization fixes that. It looks the word up in a dictionary and returns the proper base form, called the lemma.

- `ran` → `run`
- `better` → `good`
- `were` → `be`

This is slower than stemming, but it’s actually correct.

The passage’s last warning is the part people miss. Many words change role depending on the sentence:

- “I **saw** a bird.” Here `saw` is a verb, the past tense of “see”, so the lemma is `see`.
- “I cut it with a **saw**.” Here `saw` is a noun, a tool, so the lemma is `saw`.

The lemmatizer can’t tell which one it is from the word alone. You have to tell it the part of speech (noun, verb and so on). If you don’t, it assumes everything is a noun. Then `running` stays `running` instead of becoming `run`, and `were` never becomes `be`. This is what the passage means by “verbs get treated as nouns.”

### The last part: “as the same when it helps, and as different when it doesn’t”

Whether to merge words depends on the task:

- **Searching for documents about running:** treat `run`, `running` and `ran` as the same, so you find everything.
- **Telling past from present (for example, “did this already happen?”):** keep them different, because `ran` versus `run` is exactly the signal you need.

So there’s no single correct choice. You decide based on what the model needs to notice.

In [1]:
## Step 1: a regex word tokenzier
import re

def tokenzie(text):
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\sA-Za-z0-9]", text)

tokenzie("The cats weren't running at 3pm.")

['The', 'cats', "weren't", 'running', 'at', '3', 'pm', '.']

In [2]:
# Step 2: a Porter stemmer (step 1a only)

def stem_step_1a(word):
    if word.endswith("sses"):
        return word[:-2]
    if word.endswith("ies"):
        return word[:-2]
    if word.endswith("ss"):
        return word
    if word.endswith("s") and len(word) > 1:
        return word[:-1]
    return word

[stem_step_1a(w) for w in ["caresses", "ponies", "caress", "cats"]]

['caress', 'poni', 'caress', 'cat']

In [4]:
# Step 3: a lookup-based lemmatizer
LEMMA_TABLE = {
    ("running", "VERB"): "run",
    ("ran", "VERB"): "run",
    ("runs", "VERB"): "run",
    ("better", "ADJ"): "good",
    ("best", "ADJ"): "good",
    ("cats", "NOUN"): "cat",
    ("cat", "NOUN"): "cat",
    ("were", "VERB"): "be",
    ("was", "VERB"): "be",
    ("is", "VERB"): "be",
}

def lemmatize(word, pos):
    key = (word.lower(), pos)
    if key in LEMMA_TABLE:
        return LEMMA_TABLE[key]
    if pos == "VERB" and word.endswith("ing"):
        return word[:-3]
    if pos == "NOUN" and word.endswith("s"):
        return word[:-1]
    return word.lower()

lemmatize("running", "VERB")
lemmatize("cats", "NOUN")
lemmatize("better", "ADJ")


'good'

In [8]:
# In proudction we will be using NLTK and spaCY 

import nltk
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")

from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

text = "The cats were running."
tokens = word_tokenize(text)
stems = [PorterStemmer().stem(t) for t in tokens]
lemmatizer = WordNetLemmatizer()
tagged = pos_tag(tokens)


def nltk_pos_to_wordnet(tag):
    if tag.startswith("V"):
        return "v"
    if tag.startswith("J"):
        return "a"
    if tag.startswith("R"):
        return "r"
    return "n"


lemmas = [lemmatizer.lemmatize(t, nltk_pos_to_wordnet(tag)) for t, tag in tagged]
print(lemmas)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nisha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nisha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


['The', 'cat', 'be', 'run', '.']


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nisha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [11]:
!python -m spacy download en_core_web_sm

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Resolved 1 package in 3.49s
 Downloaded en-core-web-sm
Prepared 1 package in 1.88s
Installed 1 package in 338ms
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("The cats were running.")

for token in doc:
    print(token.text, token.lemma_, token.pos_)

OverflowError: Python int too large to convert to C ssize_t